In [ ]:
from collections import Counter
from typing import Optional

import stim
from IPython.core.display import Markdown

from library.qubit_allocation import QubitAllocation
from library.surface_code_patch import SurfaceCodePatch

In [ ]:
def rewrite_file_with_polygons(
    filename: str, instructions: Counter[str],
    descriptorS: tuple[SurfaceCodePatch, list[tuple[float,float]]],
    descriptorT: tuple[SurfaceCodePatch, list[tuple[float,float]]],
    descriptorM: tuple[SurfaceCodePatch, list[tuple[float,float]]]
):
    # Insert all the polygons into the Stim file for readability.
    with open(filename, "r", encoding="utf-8") as file:
        lines = file.readlines()
        inserted = 0

        surfaceS, exclusionS = descriptorS
        surfaceT, exclusionT = descriptorT
        surfaceM, exclusionM = descriptorM

        for polygon in surfaceS.get_polygons():
            lines.insert(instructions['metadata'] + inserted, polygon)
            inserted += 1
        for polygon in surfaceT.get_polygons():
            lines.insert(instructions['metadata'] + inserted, polygon)
            inserted += 1

        for polygon in surfaceS.get_polygons(exclusionS):
            lines.insert(instructions['syndrome'] + inserted, polygon)
            inserted += 1
        for polygon in surfaceT.get_polygons(exclusionT):
            lines.insert(instructions['syndrome'] + inserted, polygon)
            inserted += 1
        for polygon in surfaceM.get_polygons(exclusionM):
            lines.insert(instructions['syndrome'] + inserted, polygon)
            inserted += 1

    with open(filename, "w", encoding="utf-8") as file:
        file.writelines(lines)
        print(f"Generated circuit : {filename}")

In [ ]:
DISTANCE = 3

grid = QubitAllocation(dimensions = (7, 3))

circuit = stim.Circuit()
instructions = Counter()
grid.append_metadata(circuit)
instructions['metadata'] = len(circuit)

surfaceS = SurfaceCodePatch(distance = DISTANCE, allocation = grid, anchor = (1,1))
surfaceM = SurfaceCodePatch(distance = DISTANCE, allocation = grid, anchor = (3,1))
surfaceT = SurfaceCodePatch(distance = DISTANCE, allocation = grid, anchor = (5,1))

for moment in surfaceS.moments:
    surfaceS.append_syndrome_slice(circuit, moment, preparation = True)
    surfaceT.append_syndrome_slice(circuit, moment, preparation = True)
    circuit.append("TICK")

instructions['syndrome'] = len(circuit)

exclusion = [(3.5, 1.5), (4.5, 2.5)]
exclusionM = [(2.5, 2.5), (5.5, 1.5)]
for moment in surfaceS.moments:
    surfaceS.append_syndrome_slice(circuit, moment, preparation = False, exclusion=exclusion)
    surfaceM.append_syndrome_slice(circuit, moment, preparation = False, exclusion=exclusionM)
    surfaceT.append_syndrome_slice(circuit, moment, preparation = False, exclusion=exclusion)
    circuit.append("TICK")

instructions['merged'] = len(circuit)

print(circuit.to_crumble_url())
display(Markdown(f"[Open in Crumble]({circuit.to_crumble_url()})"))

In [ ]:
circuit.to_file("logical-teleportation.stim")
rewrite_file_with_polygons(
    "logical-teleportation.stim", instructions = instructions,
    descriptorS = (surfaceS, exclusion), descriptorT = (surfaceT, exclusion), descriptorM = (surfaceM, exclusionM)
)